# 17 — Automatic Prompt Optimization and DSPy

    ## Scenario and success criteria

    A bounded optimizer selects an extraction strategy on development cases, then receives one chance on the sealed holdout.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Define a metric before search.
- Constrain the candidate space and budget.
- Evaluate the selected candidate on untouched data.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 17 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

An optimizer can exploit metric shortcuts or memorize exposed answers.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab17 import PromptCandidate, SearchCase, after_marker, evaluate_holdout, last_token_baseline, select_on_development

cases = [
    SearchCase("city: San Francisco; state: CA", "San Francisco", "development"),
    SearchCase("city: Montréal; country: Canada", "Montréal", "development"),
    SearchCase("city: Vancouver; country: Canada", "Vancouver", "holdout"),
]
candidates = [PromptCandidate("last-token", last_token_baseline), PromptCandidate("marker", after_marker)]

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
selected = select_on_development(candidates, cases)
print("selected", selected.name)
print("holdout", evaluate_holdout(selected, cases))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert selected.name == "marker"
assert evaluate_holdout(selected, cases) == (1, 1)

## Production upgrade

Production optimizers need search budgets, traceable trials, protected holdouts, semantic-review samples, rollback, and a human-owned objective. DSPy packages this loop; it does not remove evaluation design work.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.